## Per-Patient Prediction

For selected patients, decompose each prediction through all three SAEs and visualise a geometric prediction triangle (C $\rightarrow$ P $\rightarrow$ T) alongside the sparse feature breakdown.

**Prerequisites**:
- Trained JEPA with extracted embeddings.npz
- Trained SAEs on delta, pred_error, observed_traj
- Feature cards from `inspect_sae_features` (JSON per target)
- observed_traj PCA basis from `notebooks/geometry.py`

In [ ]:
import json
import numpy as np
import matplotlib
matplotlib.use("module://matplotlib_inline.backend_inline")

from src.utils.io import EXPERIMENTS_DIR, PROCESSED_DIR, load_metadata
from src.utils.seed import load_seed, set_global_seed
from src.analysis.sae import load_sae
from src.analysis.clustering import broadcast_to_samples
from src.analysis.prediction_flow import (
    decompose_patient,
    build_patient_profile_figure,
    select_interesting_patients,
    VECTOR_NAMES,
)
from src.analysis.geometry import fit_pca

In [ ]:
# -- Config --
MODEL_TAG = "test_01"
EMB_NAME  = "embeddings_40.npz"
TOP_K_PCA = 10
SAVE_FIGS = True

exp_dir  = EXPERIMENTS_DIR / MODEL_TAG
set_global_seed(load_seed(exp_dir))

emb_path = exp_dir / "embeddings" / EMB_NAME
geo_dir  = exp_dir / "geometry"
fig_dir  = exp_dir / "prediction_flow" / "figures"

def _sp(name: str):
    return fig_dir / name if SAVE_FIGS else None

**Data Loading**

In [ ]:
# -- Load embeddings and vectors --
npz = np.load(emb_path, allow_pickle=True)
z_context   = npz["z_context"]
z_pred      = npz["z_pred"]
z_target    = npz["z_target"]
labels      = npz["labels"]
subject_ids = npz["subject_ids"]

vectors_dict = {
    "delta":         npz["delta"]         if "delta"         in npz else z_pred - z_context,
    "pred_error":    npz["pred_error"]    if "pred_error"    in npz else z_pred - z_target,
    "observed_traj": npz["observed_traj"] if "observed_traj" in npz else z_target - z_context,
}
N, D = z_context.shape
print(f"Embeddings: N={N}  D={D}")

In [ ]:
# -- Load observed_traj PCA basis (shared projection) --
pca_basis_path = geo_dir / "observed_traj_projections.npy"
if pca_basis_path.exists():
    # Refit PCA to recover the full object (projections alone aren't enough)
    pca_basis, _, _ = fit_pca(vectors_dict["observed_traj"], k=TOP_K_PCA)
    print(f"PCA basis fitted on observed_traj ({D} components)")
else:
    print(f"WARNING: {pca_basis_path} not found - fitting PCA on observed_traj")
    pca_basis, _, _ = fit_pca(vectors_dict["observed_traj"], k=TOP_K_PCA)

In [ ]:
# -- Load SAE models --
sae_models_dict: dict = {}
for target in VECTOR_NAMES:
    ckpt = exp_dir / f"sae_{target}" / "sae_checkpoint.pt"
    if ckpt.exists():
        sae_models_dict[target] = load_sae(ckpt)
        print(f"  SAE loaded: {target}")
    else:
        print(f"  SAE missing: {target} ({ckpt})")

if not sae_models_dict:
    raise FileNotFoundError("No SAE checkpoints found")

In [ ]:
# -- Load feature cards --
feature_cards_dict: dict[str, list[dict]] = {}
sae_dir = exp_dir / "sae_analysis"
for target in VECTOR_NAMES:
    cards_path = sae_dir / f"{target}_feature_cards.json"
    if cards_path.exists():
        with open(cards_path) as f:
            feature_cards_dict[target] = json.load(f)
        print(f"  Feature cards loaded: {target} ({len(feature_cards_dict[target])} features)")
    else:
        feature_cards_dict[target] = []
        print(f"  Feature cards missing: {target}")

In [ ]:
# -- Load metadata (for summary strip) --
metadata_patients, feature_names, patient_ids = load_metadata(PROCESSED_DIR)
metadata_samples = broadcast_to_samples(metadata_patients, patient_ids, subject_ids)

def _metadata_summary(idx: int) -> str:
    """One-line metadata string for patient at sample index idx."""
    row = metadata_samples[idx]
    parts = []
    for fi, name in enumerate(feature_names):
        val = row[fi]
        if name == "label":
            parts.append(f"label={int(val)}")
        elif val != 0:
            if val == int(val):
                parts.append(f"{name}={int(val)}")
            else:
                parts.append(f"{name}={val:.2f}")
    # Truncate to fit in figure strip
    summary = "  |  ".join(parts[:12])
    if len(parts) > 12:
        summary += f"  |  ... (+{len(parts)-12} more)"
    return summary

---

(phew)<br/>
### Selecting Interesting Patients

In [ ]:
"""
Picks patients across 3 criteria: 
- biggest model failures
- best predictions
- most dynamic trajectories
- and also random samples
"""
N_PATIENTS = 10

selected = select_interesting_patients(
    z_pred, z_target, z_context, labels, n=N_PATIENTS,
)

pred_error_norms = np.linalg.norm(z_pred - z_target, axis=-1)
observed_norms   = np.linalg.norm(z_target - z_context, axis=-1)
delta_norms      = np.linalg.norm(z_pred - z_context, axis=-1)

print(f"{'idx':>6}  {'label':>5}  {'||P-T||':>8}  {'||T-C||':>8}  {'||P-C||':>8}  category")
print("-" * 62)
for i, idx in enumerate(selected):
    cat = (
        "biggest failure" if i < 5 else
        "best prediction" if i < 10 else
        "most dynamic"    if i < 15 else
        "random"
    )
    print(f"{idx:>6}  {int(labels[idx]):>5}  "
          f"{pred_error_norms[idx]:>8.4f}  {observed_norms[idx]:>8.4f}  "
          f"{delta_norms[idx]:>8.4f}  {cat}")

print(f"\n{len(selected)} unique patients selected")


#### Patient Profile Figures

In [ ]:
for idx in selected:
    decomposition = decompose_patient(
        idx, vectors_dict, sae_models_dict, feature_cards_dict)

    n_active = sum(len(v) for v in decomposition.values())
    if n_active == 0:
        print(f"Patient {idx}: no active SAE features - skipping")
        continue

    fig = build_patient_profile_figure(
        patient_idx=idx,
        z_context_all=z_context,
        z_pred_all=z_pred,
        z_target_all=z_target,
        decomposition=decomposition,
        labels=labels,
        pca_basis=pca_basis,
        metadata_summary=_metadata_summary(idx),
        show=True,
        save_path=_sp(f"patient_{idx}.png"),
    )

#### Interactive Exploration

Enter a specific `PATIENT_IDX`  to generate its profile. Useful for reviewer questions or spot-checking individual cases.

In [ ]:
PATIENT_IDX = None  # <-- set to an integer index

if PATIENT_IDX is not None:
    decomposition = decompose_patient(
        PATIENT_IDX, vectors_dict, sae_models_dict, feature_cards_dict)

    print(f"Patient {PATIENT_IDX}  |  label={int(labels[PATIENT_IDX])}  |  "
          f"||P-T||={pred_error_norms[PATIENT_IDX]:.4f}  "
          f"||T-C||={observed_norms[PATIENT_IDX]:.4f}")
    print(f"Active features per vector:")
    for name in VECTOR_NAMES:
        feats = decomposition.get(name, [])
        feat_str = ", ".join(
            f"F{e['feature_idx']}({e['magnitude']:.2f})" for e in feats)
        print(f"  {name}: {feat_str or '(none)'}")

    fig = build_patient_profile_figure(
        patient_idx=PATIENT_IDX,
        z_context_all=z_context,
        z_pred_all=z_pred,
        z_target_all=z_target,
        decomposition=decomposition,
        labels=labels,
        pca_basis=pca_basis,
        metadata_summary=_metadata_summary(PATIENT_IDX),
        show=True,
        save_path=_sp(f"patient_{PATIENT_IDX}.png"),
    )